# Temperature and mortality: a causal analysis (UK)

:::{note}
**Draft scaffold.** This notebook is a work-in-progress placeholder to build out a causal analysis of the effect of temperature on mortality in the UK. See the "Plan" section below for the intended narrative, data sources, and methodology. Nothing here is final.
:::

## Motivation

The UK is unusually exposed to temperature-related mortality: a large, old housing stock with poor insulation, near-zero domestic air conditioning, and a population unaccustomed to extreme heat. Both cold winters and summer heatwaves push up deaths. This notebook builds a causal, time-series account of how temperature drives mortality, and tries to go beyond the naive "hot day → more deaths" story.

The compelling, timely hook is the **July 2022 heatwave**, when the UK recorded 40°C for the first time and the ONS attributed thousands of excess deaths to the heat episodes that summer.

This sits naturally alongside the existing [Excess deaths due to COVID-19](its_covid.ipynb) notebook, which already uses England & Wales deaths and temperature as a *control*. Here temperature is the treatment of interest rather than a nuisance variable.

## Plan (to build out)

**Research question.** What is the causal effect of temperature on UK mortality, and how much of it is explained by *accumulated* thermal exposure rather than instantaneous temperature?

**Key idea to demonstrate.** It is not just today's temperature. Homes have thermal mass: during a sustained hot spell the indoor environment keeps warming even after the outdoor peak, so a *running/cumulative* temperature term (a distributed-lag or exponentially-weighted exposure) should predict deaths better than a single-day temperature. Adding **humidity** should sharpen this further, since humid heat impairs the body's ability to cool.

**Methodology sketch.**
1. Establish an expected-mortality baseline (seasonal + trend), in the spirit of the COVID excess-deaths notebook.
2. Model deaths as a function of temperature with a **distributed lag / cumulative exposure** term (e.g. exponentially-weighted temperature, or explicit lags), so we can show the cumulative signal beats the contemporaneous one.
3. Add **humidity** and a temperature×humidity interaction.
4. Treat a specific episode (July 2022 heatwave) as an intervention and estimate excess deaths as a counterfactual, using CausalPy's ITS machinery.
5. Sensitivity / falsification: placebo-in-time checks (see the placebo notebook) to guard against spurious effects.

**Data needed (TODO).**
- **Deaths.** Ideally *daily* or *weekly* UK deaths over many years. ONS publishes weekly death registrations for England & Wales going back years; daily is harder but exists for recent years. The packaged `"covid"` dataset is currently only *monthly* (E&W, 2006–), which is too coarse to see the cumulative-heat effect — we will need a finer series.
- **Temperature.** Met Office HadUK-Grid or the Central England Temperature (CET) daily record (long, freely available). Match spatial/temporal resolution to the deaths series.
- **Humidity.** Met Office observations or ERA5 reanalysis (relative humidity). Needs sourcing and licensing check before adding to `causalpy/data/`.

**Open questions.**
- Weekly vs daily resolution: weekly is easier to source and less noisy; daily is needed to resolve short heatwaves. Decide.
- Region: England & Wales (ONS coverage) vs UK-wide. Lack of AC argues UK, but data coverage may force E&W.
- How to encode "house warms up": exponentially-weighted moving average of temperature vs explicit multi-day lags vs a simple thermal-mass ODE. Pick and justify.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import causalpy as cp

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'
seed = 42

## Starting point: the packaged monthly data

For now we load the existing `"covid"` dataset (monthly England & Wales deaths and mean temperature, 2006–) as a placeholder so the notebook runs. This is *too coarse* for the cumulative-heat story and will be replaced with a weekly/daily series (see the data TODO above).

In [ ]:
df = cp.load_data("covid")
df["date"] = pd.to_datetime(df["date"])
df.head()

In [ ]:
# Quick look: deaths and temperature move together seasonally (inverse in winter).
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(df["date"], df["deaths"])
ax[0].set(ylabel="deaths / month")
ax[1].plot(df["date"], df["temp"], color="C1")
ax[1].set(ylabel="mean temp (°C)", xlabel="date");

## Next steps

- [ ] Source a weekly (ideally daily) UK/E&W deaths series and add temperature + humidity.
- [ ] Build the seasonal baseline and the cumulative/lagged temperature exposure term.
- [ ] Add humidity and the interaction.
- [ ] Frame the July 2022 heatwave as an intervention and estimate excess deaths with CausalPy ITS.
- [ ] Placebo-in-time falsification.
- [ ] Register a thumbnail and finalise the gallery entry.